In [3]:
"""
Notes as well as a uysage guide made from AI\
This file takes the raw peptide/HLA CSV and turns it into data the PyTorch model can actually train on.

The basic biology idea is simple:

    peptide = short amino acid chain
    HLA = molecule that holds the peptide
    cancer label = whether the peptide is cancer-related or healthy

The model gets 3 views of each row:

    1. bio
       Small chemistry facts about the peptide, like weight, charge, hydrophobicity,
       stability, etc.

    2. pep
       The actual peptide amino acid letters changed into BLOSUM62 numbers.
       BLOSUM is used because amino acids are not random letters. Some amino acids
       behave more similarly than others, so BLOSUM gives the model biology-based
       numbers instead of fake number IDs.

    3. mhc
       The HLA binding groove sequence changed into BLOSUM62 numbers.
       We do not use the whole HLA protein because the binding groove is the main
       part that holds the peptide.

The target label is:

    is_cancer
        1 = cancer
        0 = healthy


The file works in this order:

    get_hla_library()
        Stores the HLA alleles the model knows.
        Each allele has its full HLA protein sequence.

    encode_blosum(seq)
        Turns amino acid letters into BLOSUM62 number rows.
        A peptide like "SLYNTVATL" becomes a matrix with one biology-number row
        per amino acid.

    hla_pseudosequence(allele, hla_library)
        Takes one HLA allele and cuts out only the important binding-groove positions.
        This makes the HLA input smaller and more focused on peptide binding.

    make_pseudo_map(hla_library)
        Builds a quick lookup table:
            HLA allele name -> HLA groove pseudosequence
        This lets each dataset row quickly find its HLA sequence.

    load_and_filter_data(path, hla_library)
        Loads the CSV.
        Drops columns that can leak the answer or act like IDs.
        Keeps only rows that have:
            is_cancer
            Peptide Sequence
            Best HLA Allele
        Also removes rows with HLA alleles not in the HLA library.

    make_splits(df)
        Splits the cleaned data into train, validation, and test.
        Stratify keeps cancer/healthy balance similar in each split.

    get_numeric_bio_candidates(train_df)
        Finds numeric peptide feature columns that are allowed to be used.
        It skips the label, peptide sequence, and HLA allele because those are handled separately.

    clean_numeric_frame(frame, columns, fill_values=None)
        Fixes missing or infinity values in numeric bio features.
        Training data creates the fill values.
        Validation/test reuse training fill values so we do not peek at them.

    select_top_bio_features(train_df)
        Uses Random Forest only to rank the peptide chemistry features.
        Keeps the top 8 strongest bio features.
        Random Forest is not the final model. It is just choosing useful features.

    make_bio_scaler(train_df, top_features)
        Scales the top bio features so big-number columns do not overpower small-number columns.

    PeptideMHCDataset
        Converts one dataframe row into PyTorch tensors:
            bio       = scaled top 8 chemistry features
            pep       = padded peptide BLOSUM matrix
            pep_mask  = tells attention what is real amino acid vs padding
            mhc       = HLA groove BLOSUM matrix
            label     = cancer/healthy

    make_datasets()
        Turns train_df, val_df, and test_df into PyTorch Dataset objects.

    make_loaders()
        Turns the Dataset objects into DataLoaders.
        These make batches for training.

    check_batch()
        Grabs one batch and checks the shapes.

    build_preprocessing_pipeline()
        Runs everything above in the correct order and returns all useful objects.


After this file runs, the main things used for training are:

    train_loader
    val_loader
    test_loader

A batch should look like:

    bio       -> (batch_size, 8)
    pep       -> (batch_size, 11, 20)
    pep_mask  -> (batch_size, 11)
    mhc       -> (batch_size, 34, 20)
    label     -> (batch_size,)

So the whole point is:

    raw CSV
        -> clean rows
        -> remove leaky columns
        -> split data
        -> choose useful peptide chemistry features
        -> encode peptide letters with BLOSUM
        -> encode HLA binding groove with BLOSUM
        -> build PyTorch batches
        -> ready for the model
"""

'\nNotes as well as a uysage guide made from AIThis file takes the raw peptide/HLA CSV and turns it into data the PyTorch model can actually train on.\n\nThe basic biology idea is simple:\n\n    peptide = short amino acid chain\n    HLA = molecule that holds the peptide\n    cancer label = whether the peptide is cancer-related or healthy\n\nThe model gets 3 views of each row:\n\n    1. bio\n       Small chemistry facts about the peptide, like weight, charge, hydrophobicity,\n       stability, etc.\n\n    2. pep\n       The actual peptide amino acid letters changed into BLOSUM62 numbers.\n       BLOSUM is used because amino acids are not random letters. Some amino acids\n       behave more similarly than others, so BLOSUM gives the model biology-based\n       numbers instead of fake number IDs.\n\n    3. mhc\n       The HLA binding groove sequence changed into BLOSUM62 numbers.\n       We do not use the whole HLA protein because the binding groove is the main\n       part that holds the

In [5]:
import os
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from Bio.Align import substitution_matrices
import matplotlib.pyplot as plt


DATA_PATH = "C:/Users/Waheg/Desktop/Peptides/Data/ML_Ready/IEDB_ML_ready_with_seq_with_HLA.csv"
CACHE_DIR = "C:/Users/Waheg/Desktop/Peptides/Methods/cache"
MAX_PEP_LEN = 11
N_TOP_BIO = 8
RANDOM_STATE = 42
BATCH_SIZE = 128
USE_37 = False

os.makedirs(CACHE_DIR, exist_ok=True)

def get_device():
    if torch.backends.mps.is_available():
        return torch.device("mps")
    elif torch.cuda.is_available():
        return torch.device("cuda")
    else:
        return torch.device("cpu")


device = get_device()
print(f"Using device: {device}")

torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)


def get_hla_library():
    return {
        "HLA-A*24:02": (
            "GSHSMRYFSTSVSRPGRGEPRFIAVGYVDDTQFVRFDSDAASQRMEPRAPWIEQEGPEYW"
            "DEETGKVKAHSQTDRENLRIALRYYNQSEAGSHTLQMMFGCDVGSDGRFLRGYHQYAYDG"
            "KDYIALKEDLRSWTAADMAAQITKRKWEAAHVAEQQRAYLEGTCVDGLRRYLENGKETLQ"
            "RTDPPKTHMTHHPISDHEATLRCWALGFYPAEITLTWQRDGEDQTQDTELVETRPAGDGT"
            "FQKWAAVVVPSGEEQRYTCHVQHEGLPKPLTLRW"
        ),
        "HLA-A*02:01": (
            "GSHSMRYFFTSVSRPGRGEPRFIAVGYVDDTQFVRFDSDAASQRMEPRAPWIEQEGPEYW"
            "DGETRKVKAHSQTHRVDLGTLRGYYNQSEAGSHTVQRMYGCDVGSDWRFLRGYHQYAYDG"
            "KDYIALKEDLRSWTAADMAAQTTKHKWEAAHVAEQLRAYLEGTCVEWLRRYLENGKETLQ"
            "RTDAPKTHMTHHAVSDHEATLRCWALSFYPAEITLTWQRDGEDQTQDTELVETRPAGDGT"
            "FQKWAAVVVPSGQEQRYTCHVQHEGLPKPLTLRW"
        ),
        "HLA-A*32:01": (
            "GSHSMRYFFTSVSRPGRGEPRFIAVGYVDDTQFVRFDSDAASQRMEPRAPWIEQEGPEYW"
            "DQETRNVKAHSQTDRESLRIALRYYNQSEAGSHTIQMMYGCDVGPDGRLLRGYQQDAYDG"
            "KDYIALNEDLRSWTAADMAAQITQRKWEAARVAEQLRAYLEGTCVEWLRRYLENGKETLQ"
            "RTDAPKTHMTHHAVSDHEATLRCWALSFYPAEITLTWQRDGEDQTQDTELVETRPAGDGT"
            "FQKWASVVVPSGQEQRYTCHVQHEGLPKPLTLRW"
        ),
        "HLA-B*15:10": (
            "GSHSMRYFYTAMSRPGRGEPRFISVGYVDDTQFVRFDSDAASPREEPRAPWIEQEGPEYW"
            "DRNTQICKTNTQTYRESLRNLRGYYNQSEAGSHTLQRMYGCDVGPDGRLLRGHDQYAYDG"
            "KDYIALNEDLSSWTAADTAAQITQRKWEAAREAEQLRAYLEGLCVEWLRRYLENGKETLQ"
            "RADPPKTHVTHHPISDHEATLRCWALGFYPAEITLTWQRDGEDQTQDTELVETRPAGDRT"
            "FQKWAAVVVPSGEEQRYTCHVQHEGLPKPLTLRW"
        ),
        "HLA-C*03:04": (
            "GSHSMRYFYTAVSRPGRGEPHFIAVGYVDDTQFVRFDSDAASPRGEPRAPWVEQEGPEYW"
            "DRETQKYKRQAQTDRVSLRNLRGYYNQSEAGSHIIQRMYGCDVGPDGRLLRGYDQYAYDG"
            "KDYIALNEDLRSWTAADTAAQITQRKWEAAREAEQLRAYLEGLCVEWLRRYLKNGKETLQ"
            "RAEHPKTHVTHHPVSDHEATLRCWALGFYPAEITLTWQWDGEDQTQDTELVETRPAGDGT"
            "FQKWAAVVVPSGEEQRYTCHVQHEGLPEPLTLRW"
        ),
        "HLA-C*14:02": (
            "CSHSMRYFSTSVSRPGRGEPRFIAVGYVDDTQFVRFDSDAASPRGEPRAPWVEQEGPEYW"
            "DRETQKYKRQAQTDRVSLRNLRGYYNQSEAGSHTLQWMFGCDLGPDGRLLRGYDQSAYDG"
            "KDYIALNEDLRSWTAADTAAQITQRKWEAAREAEQRRAYLEGTCVEWLRRYLENGKETLQ"
            "RAEHPKTHVTHHPVSDHEATLRCWALGFYPAEITLTWQWDGEDQTQDTELVETRPAGDGT"
            "FQKWAAVVVPSGEEQRYTCHVQHEGLPEPLTLRW"
        ),
        "HLA-B*44:02": (
            "GSHSMRYFYTAMSRPGRGEPRFITVGYVDDTLFVRFDSDATSPRKEPRAPWIEQEGPEYW"
            "DRETQISKTNTQTYRENLRTALRYYNQSEAGSHIIQRMYGCDVGPDGRLLRGYDQDAYDG"
            "KDYIALNEDLSSWTAADTAAQITQRKWEAARVAEQDRAYLEGLCVESLRRYLENGKETLQ"
            "RADPPKTHVTHHPISDHEVTLRCWALGFYPAEITLTWQRDGEDQTQDTELVETRPAGDRT"
            "FQKWAAVVVPSGEEQRYTCHVQHEGLPKPLTLRW"
        ),
        "HLA-B*57:01": (
            "GSHSMRYFYTAMSRPGRGEPRFIAVGYVDDTQFVRFDSDAASPRMAPRAPWIEQEGPEYW"
            "DGETRNMKASAQTYRENLRIALRYYNQSEAGSHIIQVMYGCDVGPDGRLLRGHDQSAYDG"
            "KDYIALNEDLSSWTAADTAAQITQRKWEAARVAEQLRAYLEGLCVEWLRRYLENGKETLQ"
            "RADPPKTHVTHHPISDHEATLRCWALGFYPAEITLTWQRDGEDQTQDTELVETRPAGDRT"
            "FQKWAAVVVPSGEEQRYTCHVQHEGLPKPLTLRW"
        ),
        "HLA-A*03:01": (
            "GSHSMRYFFTSVSRPGRGEPRFIAVGYVDDTQFVRFDSDAASQRMEPRAPWIEQEGPEYW"
            "DQETRNVKAQSQTDRVDLGTLRGYYNQSEAGSHTIQIMYGCDVGSDGRFLRGYRQDAYDG"
            "KDYIALNEDLRSWTAADMAAQITKRKWEAAHEAEQLRAYLDGTCVEWLRRYLENGKETLQ"
            "RTDPPKTHMTHHPISDHEATLRCWALGFYPAEITLTWQRDGEDQTQDTELVETRPAGDGT"
            "FQKWAAVVVPSGEEQRYTCHVQHEGLPKPLTLRW"
        ),
        "HLA-C*05:01": (
            "CSHSMRYFYTAVSRPGRGEPRFIAVGYVDDTQFVQFDSDAASPRGEPRAPWVEQEGPEYW"
            "DRETQKYKRQAQTDRVNLRKLRGYYNQSEAGSHTLQRMYGCDLGPDGRLLRGYNQFAYDG"
            "KDYIALNEDLRSWTAADKAAQITQRKWEAAREAEQRRAYLEGTCVEWLRRYLENGKKTLQ"
            "RAEHPKTHVTHHPVSDHEATLRCWALGFYPAEITLTWQRDGEDQTQDTELVETRPAGDGT"
            "FQKWAAVVVPSGEEQRYTCHVQHEGLPEPLTLRW"
        ),
        "HLA-B*27:05": (
            "GSHSMRYFHTSVSRPGRGEPRFITVGYVDDTLFVRFDSDAASPREEPRAPWIEQEGPEYW"
            "DRETQICKAKAQTDREDLRTLLRYYNQSEAGSHTLQNMYGCDVGPDGRLLRGYHQDAYDG"
            "KDYIALNEDLSSWTAADTAAQITQRKWEAARVAEQLRAYLEGECVEWLRRYLENGKETLQ"
            "RADPPKTHVTHHPISDHEATLRCWALGFYPAEITLTWQRDGEDQTQDTELVETRPAGDRT"
            "FQKWAAVVVPSGEEQRYTCHVQHEGLPKPLTLRW"
        ),
        "HLA-B*51:01": (
            "GSHSMRYFYTAMSRPGRGEPRFIAVGYVDDTQFVRFDSDAASPRTEPRAPWIEQEGPEYW"
            "DRNTQIFKTNTQTYRENLRIALRYYNQSEAGSHTWQTMYGCDVGPDGRLLRGHNQYAYDG"
            "KDYIALNEDLSSWTAADTAAQITQRKWEAAREAEQLRAYLEGLCVEWLRRHLENGKETLQ"
            "RADPPKTHVTHHPVSDHEATLRCWALGFYPAEITLTWQRDGEDQTQDTELVETRPAGDRT"
            "FQKWAAVVVPSGEEQRYTCHVQHEGLPKPLTLRW"
        ),
        "HLA-B*40:01": (
            "GSHSMRYFHTAMSRPGRGEPRFITVGYVDDTLFVRFDSDATSPRKEPRAPWIEQEGPEYW"
            "DRETQISKTNTQTYRESLRNLRGYYNQSEAGSHTLQRMYGCDVGPDGRLLRGHNQYAYDG"
            "KDYIALNEDLRSWTAADTAAQISQRKLEAARVAEQLRAYLEGECVEWLRRYLENGKDKLE"
            "RADPPKTHVTHHPISDHEATLRCWALGFYPAEITLTWQRDGEDQTQDTELVETRPAGDRT"
            "FQKWAAVVVPSGEEQRYTCHVQHEGLPKPLTLRW"
        ),
        "HLA-B*40:02": (
            "GSHSMRYFHTSVSRPGRGEPRFITVGYVDDTLFVRFDSDATSPRKEPRAPWIEQEGPEYW"
            "DRETQISKTNTQTYRESLRNLRGYYNQSEAGSHTLQSMYGCDVGPDGRLLRGHNQYAYDG"
            "KDYIALNEDLRSWTAADTAAQITQRKWEAARVAEQLRAYLEGECVEWLRRYLENGKETLQ"
            "RADPPKTHVTHHPISDHEATLRCWALGFYPAEITLTWQRDGEDQTQDTELVETRPAGDRT"
            "FQKWAAVVVPSGEEQRYTCHVQHEGLPKPLTLRW"
        ),
        "HLA-B*07:02": (
            "GSHSMRYFYTSVSRPGRGEPRFISVGYVDDTQFVRFDSDAASPREEPRAPWIEQEGPEYW"
            "DRNTQIYKAQAQTDRESLRNLRGYYNQSEAGSHTLQSMYGCDVGPDGRLLRGHDQYAYDG"
            "KDYIALNEDLRSWTAADTAAQITQRKWEAAREAEQRRAYLEGECVEWLRRYLENGKDKLE"
            "RADPPKTHVTHHPISDHEATLRCWALGFYPAEITLTWQRDGEDQTQDTELVETRPAGDRT"
            "FQKWAAVVVPSGEEQRYTCHVQHEGLPKPLTLRW"
        ),
        "HLA-B*18:01": (
            "GSHSMRYFHTSVSRPGRGEPRFISVGYVDGTQFVRFDSDAASPRTEPRAPWIEQEGPEYW"
            "DRNTQISKTNTQTYRESLRNLRGYYNQSEAGSHTLQRMYGCDVGPDGRLLRGHDQSAYDG"
            "KDYIALNEDLSSWTAADTAAQITQRKWEAARVAEQLRAYLEGTCVEWLRRHLENGKETLQ"
            "RADPPKTHVTHHPISDHEATLRCWALGFYPAEITLTWQRDGEDQTQDTELVETRPAGDRT"
            "FQKWAAVVVPSGEEQRYTCHVQHEGLPKPLTLRW"
        ),
        "HLA-A*25:01": (
            "GSHSMRYFYTSVSRPGRGEPRFIAVGYVDDTQFVRFDSDAASQRMEPRAPWIEQEGPEYW"
            "DRNTRNVKAHSQTDRESLRIALRYYNQSEDGSHTIQRMYGCDVGPDGRFLRGYQQDAYDG"
            "KDYIALNEDLRSWTAADMAAQITQRKWETAHEAEQWRAYLEGRCVEWLRRYLENGKETLQ"
            "RTDAPKTHMTHHAVSDHEATLRCWALSFYPAEITLTWQRDGEDQTQDTELVETRPAGDGT"
            "FQKWASVVVPSGQEQRYTCHVQHEGLPKPLTLRW"
        ),
        "HLA-B*44:03": (
            "GSHSMRYFYTAMSRPGRGEPRFITVGYVDDTLFVRFDSDATSPRKEPRAPWIEQEGPEYW"
            "DRETQISKTNTQTYRENLRTALRYYNQSEAGSHIIQRMYGCDVGPDGRLLRGYDQDAYDG"
            "KDYIALNEDLSSWTAADTAAQITQRKWEAARVAEQLRAYLEGLCVESLRRYLENGKETLQ"
            "RADPPKTHVTHHPISDHEVTLRCWALGFYPAEITLTWQRDGEDQTQDTELVETRPAGDRT"
            "FQKWAAVVVPSGEEQRYTCHVQHEGLPKPLTLRW"
        ),
        "HLA-A*30:01": (
            "GSHSMRYFSTSVSRPGSGEPRFIAVGYVDDTQFVRFDSDAASQRMEPRAPWIEQERPEYW"
            "DQETRNVKAQSQTDRVDLGTLRGYYNQSEAGSHTIQIMYGCDVGSDGRFLRGYEQHAYDG"
            "KDYIALNEDLRSWTAADMAAQITQRKWEAARWAEQLRAYLEGTCVEWLRRYLENGKETLQ"
            "RTDPPKTHMTHHPISDHEATLRCWALGFYPAEITLTWQRDGEDQTQDTELVETRPAGDGT"
            "FQKWAAVVVPSGEEQRYTCHVQHEGLPKPLTLRW"
        ),
        "HLA-A*11:01": (
            "GSHSMRYFYTSVSRPGRGEPRFIAVGYVDDTQFVRFDSDAASQRMEPRAPWIEQEGPEYW"
            "DQETRNVKAQSQTDRVDLGTLRGYYNQSEDGSHTIQIMYGCDVGPDGRFLRGYRQDAYDG"
            "KDYIALNEDLRSWTAADMAAQITKRKWEAAHAAEQQRAYLEGRCVEWLRRYLENGKETLQ"
            "RTDPPKTHMTHHPISDHEATLRCWALGFYPAEITLTWQRDGEDQTQDTELVETRPAGDGT"
            "FQKWAAVVVPSGEEQRYTCHVQHEGLPKPLTLRW"
        ),
        "HLA-C*12:03": (
            "CSHSMRYFYTAVSRPGRGEPRFIAVGYVDDTQFVRFDSDAASPRGEPRAPWVEQEGPEYW"
            "DRETQKYKRQAQADRVSLRNLRGYYNQSEAGSHTLQWMYGCDLGPDGRLLRGYDQSAYDG"
            "KDYIALNEDLRSWTAADTAAQITQRKWEAAREAEQWRAYLEGTCVEWLRRYLENGKETLQ"
            "RAEHPKTHVTHHPVSDHEATLRCWALGFYPAEITLTWQRDGEDQTQDTELVETRPAGDGT"
            "FQKWAAVVVPSGEEQRYTCHVQHEGLPEPLTLRW"
        ),
        "HLA-B*49:01": (
            "GSHSMRYFHTAMSRPGRGEPRFITVGYVDDTLFVRFDSDATSPRKEPRAPWIEQEGPEYW"
            "DRETQISKTNTQTYRENLRIALRYYNQSEAGSHTWQRMYGCDLGPDGRLLRGYNQLAYDG"
            "KDYIALNEDLSSWTAADTAAQITQRKWEAAREAEQLRAYLEGLCVEWLRRYLENGKETLQ"
            "RADPPKTHVTHHPISDHEATLRCWALGFYPAEITLTWQRDGEDQTQDTELVETRPAGDRT"
            "FQKWAAVVVPSGEEQRYTCHVQHEGLPKPLTLRW"
        ),
        "HLA-A*01:01": (
            "GSHSMRYFFTSVSRPGRGEPRFIAVGYVDDTQFVRFDSDAASQKMEPRAPWIEQEGPEYW"
            "DQETRNMKAHSQTDRANLGTLRGYYNQSEDGSHTIQIMYGCDVGPDGRFLRGYRQDAYDG"
            "KDYIALNEDLRSWTAADMAAQITKRKWEAVHAAEQRRVYLEGRCVDGLRRYLENGKETLQ"
            "RTDPPKTHMTHHPISDHEATLRCWALGFYPAEITLTWQRDGEDQTQDTELVETRPAGDGT"
            "FQKWAAVVVPSGEEQRYTCHVQHEGLPKPLTLRW"
        ),
        "HLA-B*15:02": (
            "GSHSMRYFYTAMSRPGRGEPRFIAVGYVDDTQFVRFDSDAASPRMAPRAPWIEQEGPEYW"
            "DRNTQISKTNTQTYRESLRNLRGYYNQSEAGSHIIQRMYGCDVGPDGRLLRGYDQSAYDG"
            "KDYIALNEDLSSWTAADTAAQITQRKWEAAREAEQLRAYLEGLCVEWLRRYLENGKETLQ"
            "RADPPKTHVTHHPISDHEATLRCWALGFYPAEITLTWQRDGEDQTQDTELVETRPAGDRT"
            "FQKWAAVVVPSGEEQRYTCHVQHEGLPKPLTLRW"
        ),
        "HLA-B*35:01": (
            "GSHSMRYFYTAMSRPGRGEPRFIAVGYVDDTQFVRFDSDAASPRTEPRAPWIEQEGPEYW"
            "DRNTQIFKTNTQTYRESLRNLRGYYNQSEAGSHIIQRMYGCDLGPDGRLLRGHDQSAYDG"
            "KDYIALNEDLSSWTAADTAAQITQRKWEAARVAEQLRAYLEGLCVEWLRRYLENGKETLQ"
            "RADPPKTHVTHHPVSDHEATLRCWALGFYPAEITLTWQRDGEDQTQDTELVETRPAGDRT"
            "FQKWAAVVVPSGEEQRYTCHVQHEGLPKPLTLRW"
        ),
        "HLA-A*68:02": (
            "GSHSMRYFYTSMSRPGRGEPRFIAVGYVDDTQFVRFDSDAASQRMEPRAPWIEQEGPEYW"
            "DRNTRNVKAQSQTDRVDLGTLRGYYNQSEAGSHTIQRMYGCDVGPDGRFLRGYHQYAYDG"
            "KDYIALKEDLRSWTAADMAAQTTKHKWEAAHVAEQWRAYLEGTCVEWLRRYLENGKETLQ"
            "RTDAPKTHMTHHAVSDHEATLRCWALSFYPAEITLTWQRDGEDQTQDTELVETRPAGDGT"
            "FQKWVAVVVPSGQEQRYTCHVQHEGLPKPLTLRW"
        ),
        "HLA-B*15:01": (
            "GSHSMRYFYTAMSRPGRGEPRFIAVGYVDDTQFVRFDSDAASPRMAPRAPWIEQEGPEYW"
            "DRETQISKTNTQTYRESLRNLRGYYNQSEAGSHTLQRMYGCDVGPDGRLLRGHDQSAYDG"
            "KDYIALNEDLSSWTAADTAAQITQRKWEAAREAEQWRAYLEGLCVEWLRRYLENGKETLQ"
            "RADPPKTHVTHHPISDHEATLRCWALGFYPAEITLTWQRDGEDQTQDTELVETRPAGDRT"
            "FQKWAAVVVPSGEEQRYTCHVQHEGLPKPLTLRW"
        ),
    }


_AAS = list("ACDEFGHIKLMNPQRSTVWY")
_BLOSUM62 = substitution_matrices.load("BLOSUM62")

NETMHCPAN_POS34 = [
    7, 9, 24, 45, 59, 62, 63, 66, 67, 69, 70, 73, 74, 76, 77, 80, 81, 84,
    95, 97, 99, 114, 116, 118, 143, 147, 150, 152, 156, 158, 159, 163, 167, 171
]

MHCFLURRY_EXTRA = [66, 76, 9]
PSEUDO_POS = NETMHCPAN_POS34 + (MHCFLURRY_EXTRA if USE_37 else [])


def encode_blosum(seq):
    rows = []
    for aa in str(seq).upper():
        if aa in _AAS:
            rows.append([float(_BLOSUM62[aa][other]) for other in _AAS])
        else:
            rows.append([0.0] * 20)
    return np.array(rows, dtype=np.float32)


def hla_pseudosequence(allele, hla_library):
    seq = hla_library[allele]
    return "".join(seq[p - 1] if p - 1 < len(seq) else "-" for p in PSEUDO_POS)


def make_pseudo_map(hla_library):
    pseudo_map = {allele: hla_pseudosequence(allele, hla_library) for allele in hla_library}
    if len(set(pseudo_map.values())) != len(pseudo_map):
        raise ValueError("Pseudosequence collision found. Adjust PSEUDO_POS.")
    return pseudo_map


def load_and_filter_data(path, hla_library):
    df = pd.read_csv(path, low_memory=False)

    leaky_cols = [
        "Disease", "strat_key", "found_in_healthy", "ID", "Tissue",
        "hla_population_freq", "Uniprot IDs", "source", "hla_source",
        "hla_known", "feature_error", "Affinity % Rank", "affinity_bin",
        "is_strong_binder", "is_weak_binder", "is_non_binder",
        "ms_confirmed", "MHC Class", "Peptide Modifications", "length_class"
    ]

    df = df.drop(columns=leaky_cols, errors="ignore")
    df = df.dropna(subset=["is_cancer", "Peptide Sequence", "Best HLA Allele"])
    df = df[df["Best HLA Allele"].isin(hla_library.keys())].reset_index(drop=True)
    df["is_cancer"] = df["is_cancer"].astype(int)

    return df


def make_splits(df):
    train_df, test_df = train_test_split(
        df,
        test_size=0.15,
        random_state=RANDOM_STATE,
        stratify=df["is_cancer"]
    )

    train_df, val_df = train_test_split(
        train_df,
        test_size=0.1765,
        random_state=RANDOM_STATE,
        stratify=train_df["is_cancer"]
    )

    return (
        train_df.reset_index(drop=True),
        val_df.reset_index(drop=True),
        test_df.reset_index(drop=True)
    )


def get_numeric_bio_candidates(train_df):
    exclude = {"is_cancer", "Peptide Sequence", "Best HLA Allele"}
    return [
        c for c in train_df.columns
        if c not in exclude and pd.api.types.is_numeric_dtype(train_df[c])
    ]


def clean_numeric_frame(frame, columns, fill_values=None):
    x = frame[columns].replace([np.inf, -np.inf], np.nan)

    if fill_values is None:
        fill_values = x.mean(numeric_only=True).fillna(0)

    x = x.fillna(fill_values).fillna(0)
    return x, fill_values


def select_top_bio_features(train_df, n_top=N_TOP_BIO):
    bio_candidates = get_numeric_bio_candidates(train_df)
    x_train, fill_values = clean_numeric_frame(train_df, bio_candidates)

    rf = RandomForestClassifier(
        n_estimators=200,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

    rf.fit(x_train, train_df["is_cancer"])

    importances = pd.Series(
        rf.feature_importances_,
        index=bio_candidates
    ).sort_values(ascending=False)

    top_features = importances.head(n_top).index.tolist()

    return top_features, importances, fill_values


def make_bio_scaler(train_df, top_features):
    x_train, fill_values = clean_numeric_frame(train_df, top_features)
    scaler = StandardScaler().fit(x_train.values)
    return scaler, fill_values


def plot_top_features(importances, n_top=N_TOP_BIO):
    plt.figure(figsize=(8, 4))
    importances.head(n_top)[::-1].plot(kind="barh")
    plt.title(f"Top {n_top} biophysical features")
    plt.tight_layout()
    plt.show()


def visualize_row(frame, idx, top_features, pseudo_map):
    row = frame.iloc[idx]

    pep = row["Peptide Sequence"]
    allele = row["Best HLA Allele"]
    label = int(row["is_cancer"])

    pep_enc = encode_blosum(pep)
    pseq = pseudo_map[allele]
    mhc_enc = encode_blosum(pseq)
    bio_vals = row[top_features].astype(float).values

    fig = plt.figure(figsize=(18, 4.5))
    fig.suptitle(
        f"row {idx} | {pep} | {allele} | {'CANCER' if label else 'HEALTHY'}",
        fontweight="bold"
    )

    ax1 = fig.add_subplot(1, 3, 1)
    ax1.barh(top_features[::-1], bio_vals[::-1])
    ax1.set_title("Stage 1 - biophysical")
    ax1.tick_params(axis="y", labelsize=8)

    ax2 = fig.add_subplot(1, 3, 2)
    im2 = ax2.imshow(pep_enc.T, aspect="auto", cmap="RdBu_r")
    ax2.set_yticks(range(20))
    ax2.set_yticklabels(_AAS, fontsize=6)
    ax2.set_xticks(range(len(pep)))
    ax2.set_xticklabels(list(pep), fontsize=8)
    ax2.set_title(f"Stage 2 - peptide BLOSUM {pep_enc.shape}")
    fig.colorbar(im2, ax=ax2, shrink=0.7)

    ax3 = fig.add_subplot(1, 3, 3)
    im3 = ax3.imshow(mhc_enc.T, aspect="auto", cmap="PuOr_r")
    ax3.set_yticks(range(20))
    ax3.set_yticklabels(_AAS, fontsize=6)
    ax3.set_xticks(range(len(pseq)))
    ax3.set_xticklabels(list(pseq), fontsize=6)
    ax3.set_title(f"Stage 3 - HLA BLOSUM {mhc_enc.shape}")
    fig.colorbar(im3, ax=ax3, shrink=0.7)

    plt.tight_layout()
    plt.show()


class PeptideMHCDataset(Dataset):
    def __init__(self, frame, top_features, scaler, fill_values, pseudo_map):
        self.peptides = frame["Peptide Sequence"].tolist()
        self.alleles = frame["Best HLA Allele"].tolist()
        self.labels = frame["is_cancer"].astype(int).values
        self.pseudo_map = pseudo_map

        bio = frame[top_features].replace([np.inf, -np.inf], np.nan)
        bio = bio.fillna(fill_values).fillna(0).values
        self.bio = scaler.transform(bio).astype(np.float32)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        pep_enc = encode_blosum(self.peptides[idx])
        pep_len = pep_enc.shape[0]

        pep_pad = np.zeros((MAX_PEP_LEN, 20), dtype=np.float32)
        pep_pad[:min(pep_len, MAX_PEP_LEN)] = pep_enc[:MAX_PEP_LEN]

        pep_mask = np.ones(MAX_PEP_LEN, dtype=bool)
        pep_mask[:min(pep_len, MAX_PEP_LEN)] = False

        mhc_enc = encode_blosum(self.pseudo_map[self.alleles[idx]])

        return {
            "bio": torch.from_numpy(self.bio[idx]),
            "pep": torch.from_numpy(pep_pad),
            "pep_mask": torch.from_numpy(pep_mask),
            "mhc": torch.from_numpy(mhc_enc),
            "label": torch.tensor(self.labels[idx], dtype=torch.long),
        }


def make_datasets(train_df, val_df, test_df, top_features, scaler, fill_values, pseudo_map):
    train_ds = PeptideMHCDataset(train_df, top_features, scaler, fill_values, pseudo_map)
    val_ds = PeptideMHCDataset(val_df, top_features, scaler, fill_values, pseudo_map)
    test_ds = PeptideMHCDataset(test_df, top_features, scaler, fill_values, pseudo_map)

    return train_ds, val_ds, test_ds


def make_loaders(train_ds, val_ds, test_ds, batch_size=BATCH_SIZE):
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False)

    return train_loader, val_loader, test_loader


def check_batch(loader):
    batch = next(iter(loader))
    shapes = {k: tuple(v.shape) for k, v in batch.items()}
    return shapes


def build_preprocessing_pipeline():
    hla_library = get_hla_library()
    pseudo_map = make_pseudo_map(hla_library)

    df = load_and_filter_data(DATA_PATH, hla_library)
    train_df, val_df, test_df = make_splits(df)

    top_features, importances, _ = select_top_bio_features(train_df, N_TOP_BIO)
    bio_scaler, bio_fill_values = make_bio_scaler(train_df, top_features)

    train_ds, val_ds, test_ds = make_datasets(
        train_df,
        val_df,
        test_df,
        top_features,
        bio_scaler,
        bio_fill_values,
        pseudo_map
    )

    train_loader, val_loader, test_loader = make_loaders(
        train_ds,
        val_ds,
        test_ds,
        BATCH_SIZE
    )

    print(f"Device: {device}")
    print(f"Rows: {len(df):,}")
    print(f"Alleles used: {df['Best HLA Allele'].nunique()}")
    print(f"Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")
    print(f"Top {N_TOP_BIO} features: {top_features}")
    print(f"Batch shapes: {check_batch(train_loader)}")

    return {
        "df": df,
        "train_df": train_df,
        "val_df": val_df,
        "test_df": test_df,
        "top_features": top_features,
        "feature_importances": importances,
        "bio_scaler": bio_scaler,
        "bio_fill_values": bio_fill_values,
        "hla_library": hla_library,
        "pseudo_map": pseudo_map,
        "train_ds": train_ds,
        "val_ds": val_ds,
        "test_ds": test_ds,
        "train_loader": train_loader,
        "val_loader": val_loader,
        "test_loader": test_loader,
        "device": device,
        "pseudo_len": len(PSEUDO_POS),
    }


artifacts = build_preprocessing_pipeline()

df = artifacts["df"]
train_df = artifacts["train_df"]
val_df = artifacts["val_df"]
test_df = artifacts["test_df"]

TOP8 = artifacts["top_features"]
feature_importances = artifacts["feature_importances"]

bio_scaler = artifacts["bio_scaler"]
hla_library = artifacts["hla_library"]
pseudo_map = artifacts["pseudo_map"]

train_ds = artifacts["train_ds"]
val_ds = artifacts["val_ds"]
test_ds = artifacts["test_ds"]

train_loader = artifacts["train_loader"]
val_loader = artifacts["val_loader"]
test_loader = artifacts["test_loader"]

PSEUDO_LEN = artifacts["pseudo_len"]

Using device: cpu
Device: cpu
Rows: 43,678
Alleles used: 27
Train: 30,573 | Val: 6,553 | Test: 6,552
Top 8 features: ['molecular_weight', 'tcr_contact_hydro_mean', 'boman_index', 'hydrophobicity_GRAVY', 'instability_index', 'charge_pH_7', 'isoelectric_point', 'aliphatic_index']
Batch shapes: {'bio': (128, 8), 'pep': (128, 11, 20), 'pep_mask': (128, 11), 'mhc': (128, 34, 20), 'label': (128,)}
